In [1]:
!pip install ultralytics

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import zipfile
import os


zip_file_path = 'Fruits by YOLO.v1i.yolov8.zip' 
extract_folder = 'dataset'  


os.makedirs(extract_folder, exist_ok=True)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)


print('Extracted files:', os.listdir(extract_folder))


dataset_dir = os.path.join(extract_folder) 
print('Dataset structure:', os.listdir(dataset_dir))


Extracted files: ['data.yaml', 'README.dataset.txt', 'README.roboflow.txt', 'test', 'train', 'valid']
Dataset structure: ['data.yaml', 'README.dataset.txt', 'README.roboflow.txt', 'test', 'train', 'valid']


In [4]:
from collections import Counter
from pathlib import Path

# EDA: count split sizes, class distribution, and label statistics
label_paths = list(Path(dataset_dir).rglob('labels/*.txt'))
image_paths = list(Path(dataset_dir).rglob('images/*'))

print(f'Total image files found: {len(image_paths)}')
print(f'Total label files found: {len(label_paths)}')

splits = {}
for split_path in Path(dataset_dir).iterdir():
    if split_path.is_dir():
        image_files = list(split_path.glob('images/*'))
        label_files = list(split_path.glob('labels/*.txt'))
        if image_files or label_files:
            splits[split_path.name] = {
                'images': len(image_files),
                'labels': len(label_files)
            }

print('\nSplit summary:')
for split_name, stats in splits.items():
    print(f'  {split_name}: {stats["images"]} images, {stats["labels"]} labels')

class_counter = Counter()
box_counts = []
box_sizes = []

for label_path in label_paths:
    with open(label_path, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f if line.strip()]

    box_counts.append(len(lines))
    for line in lines:
        parts = line.split()
        if len(parts) >= 5:
            class_id = int(parts[0])
            width = float(parts[3])
            height = float(parts[4])
            class_counter[class_id] += 1
            box_sizes.append((width, height))

print(f'\nAverage boxes per labeled image: {sum(box_counts) / len(box_counts):.2f}')
print(f'Total boxes: {sum(box_counts)}')

if class_counter:
    print('\nClass distribution:')
    for class_id, count in class_counter.most_common():
        print(f'  class {class_id}: {count} boxes')


# show top 5 most common classes
if class_counter:
    most_common = class_counter.most_common(5)
    print('\nTop 5 classes:')
    for class_id, count in most_common:
        print(f'  class {class_id}: {count}')


Total image files found: 2974
Total label files found: 2974

Split summary:
  test: 90 images, 90 labels
  train: 2697 images, 2697 labels
  valid: 187 images, 187 labels

Average boxes per labeled image: 2.14
Total boxes: 6362

Class distribution:
  class 5: 990 boxes
  class 8: 890 boxes
  class 7: 828 boxes
  class 3: 815 boxes
  class 4: 762 boxes
  class 1: 637 boxes
  class 0: 547 boxes
  class 2: 527 boxes
  class 6: 366 boxes

Top 5 classes:
  class 5: 990
  class 8: 890
  class 7: 828
  class 3: 815
  class 4: 762


In [16]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

results = model.train(
    data='dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=8,
    workers=2,
    cache=False,
    project='runs/train',
    name='yolo11_fruits',
    exist_ok=True
)

best_model_path = 'runs/train/yolo11_fruits/weights/best.pt'

best_model = YOLO(best_model_path)

metrics = best_model.val(
    data='dataset/data.yaml',
    workers=2
)

precision = metrics.box.mp
recall = metrics.box.mr
map50 = metrics.box.map50
map50_95 = metrics.box.map

f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"Mean Precision: {precision:.4f}")
print(f"Mean Recall: {recall:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map50_95:.4f}")
print(f"F1-score: {f1_score:.4f}")
print(f"Best model: {best_model_path}")

New https://pypi.org/project/ultralytics/8.4.48 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.203  Python-3.12.6 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3050 Ti Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolo11_fruits, nbs=64, nm